In [2]:
import pandas as pd
import numpy as np

RAW_DIR = "../data/raw"

# 1. Load events and category_tree
events = pd.read_csv(f"{RAW_DIR}/events.csv")
cat_tree = pd.read_csv(f"{RAW_DIR}/category_tree.csv")

# Convert 13-digit Unix ms timestamps to UTC datetime
events["dt"] = pd.to_datetime(events["timestamp"], unit="ms", utc=True)

print("=== EVENTS SCHEMA & SUMMARY ===")
print(f"Total rows: {len(events):,}")
print(f"Unique visitors: {events['visitorid'].nunique():,}")
print(f"Unique items: {events['itemid'].nunique():,}")
print(f"Date range: {events['dt'].min()}  -->  {events['dt'].max()}")
print("\nMissing values per column:\n", events.isna().sum())
print("\nEvent counts:\n", events["event"].value_counts())

print("\n=== CATEGORY TREE ===")
print(f"Total categories: {len(cat_tree):,}")
print(cat_tree.head(10))

# 2. Sample item_properties to inspect hashed attributes (loading full 20M+ rows in chunks/sample first)
props_sample = pd.read_csv(f"{RAW_DIR}/item_properties_part1.csv", nrows=500_000)
print("\n=== ITEM PROPERTIES (FIRST 500K ROWS SAMPLE) ===")
print("Columns:", props_sample.columns.tolist())
print("Top 15 most frequent property names:")
print(props_sample["property"].value_counts().head(15))
print("\nSample rows:")
print(props_sample.head(10))

=== EVENTS SCHEMA & SUMMARY ===
Total rows: 2,756,101
Unique visitors: 1,407,580
Unique items: 235,061
Date range: 2015-05-03 03:00:04.384000+00:00  -->  2015-09-18 02:59:47.788000+00:00

Missing values per column:
 timestamp              0
visitorid              0
event                  0
itemid                 0
transactionid    2733644
dt                     0
dtype: int64

Event counts:
 event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64

=== CATEGORY TREE ===
Total categories: 1,669
   categoryid  parentid
0        1016     213.0
1         809     169.0
2         570       9.0
3        1691     885.0
4         536    1691.0
5         231       NaN
6         542     378.0
7        1146     542.0
8        1140     542.0
9        1479    1537.0

=== ITEM PROPERTIES (FIRST 500K ROWS SAMPLE) ===
Columns: ['timestamp', 'itemid', 'property', 'value']
Top 15 most frequent property names:
property
888           73919
790           44241
ava

In [3]:
# 1. Overall Event & Visitor Funnel
event_counts = events["event"].value_counts()
visitor_funnel = events.groupby("event")["visitorid"].nunique()

total_visitors = events["visitorid"].nunique()
view_visitors = visitor_funnel.get("view", 0)
cart_visitors = visitor_funnel.get("addtocart", 0)
txn_visitors = visitor_funnel.get("transaction", 0)

print("=== VISITOR-LEVEL CONVERSION FUNNEL ===")
print(f"Total Unique Visitors:      {total_visitors:,} (100.00%)")
print(f"Visitors with >=1 View:     {view_visitors:,} ({view_visitors/total_visitors:.2%})")
print(f"Visitors with >=1 Cart Add: {cart_visitors:,} ({cart_visitors/total_visitors:.2%})")
print(f"Visitors with >=1 Purchase: {txn_visitors:,} ({txn_visitors/total_visitors:.2%})")
print(f"\nView -> Add-to-Cart Rate (User-level): {cart_visitors/view_visitors:.2%}")
print(f"Add-to-Cart -> Purchase Rate (User-level): {txn_visitors/cart_visitors:.2%}")
print(f"Overall View -> Purchase Rate (User-level): {txn_visitors/view_visitors:.2%}")

# 2. User Interaction Sparsity (Crucial for Cold-Start & Segmentation)
user_activity = events.groupby("visitorid").size()
print("\n=== USER INTERACTION DISTRIBUTION ===")
print(user_activity.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))
print(f"\nSingle-event visitors (1 interaction only): {(user_activity == 1).mean():.2%}")
print(f"Visitors with >= 3 interactions: {(user_activity >= 3).mean():.2%}")
print(f"Visitors with >= 5 interactions: {(user_activity >= 5).mean():.2%}")

=== VISITOR-LEVEL CONVERSION FUNNEL ===
Total Unique Visitors:      1,407,580 (100.00%)
Visitors with >=1 View:     1,404,179 (99.76%)
Visitors with >=1 Cart Add: 37,722 (2.68%)
Visitors with >=1 Purchase: 11,719 (0.83%)

View -> Add-to-Cart Rate (User-level): 2.69%
Add-to-Cart -> Purchase Rate (User-level): 31.07%
Overall View -> Purchase Rate (User-level): 0.83%

=== USER INTERACTION DISTRIBUTION ===
count    1.407580e+06
mean     1.958042e+00
std      1.258049e+01
min      1.000000e+00
50%      1.000000e+00
75%      2.000000e+00
90%      3.000000e+00
95%      5.000000e+00
99%      1.300000e+01
max      7.757000e+03
dtype: float64

Single-event visitors (1 interaction only): 71.15%
Visitors with >= 3 interactions: 14.21%
Visitors with >= 5 interactions: 5.80%


In [4]:
# 1. Visitor-Level Conversion Funnel
visitor_funnel = events.groupby("event")["visitorid"].nunique()
total_visitors = events["visitorid"].nunique()
view_visitors = visitor_funnel.get("view", 0)
cart_visitors = visitor_funnel.get("addtocart", 0)
txn_visitors = visitor_funnel.get("transaction", 0)

print("=== VISITOR-LEVEL CONVERSION FUNNEL ===")
print(f"Total Unique Visitors:         {total_visitors:,} (100.00%)")
print(f"Visitors with >=1 View:        {view_visitors:,} ({view_visitors/total_visitors:.2%})")
print(f"Visitors with >=1 Add-to-Cart: {cart_visitors:,} ({cart_visitors/total_visitors:.2%})")
print(f"Visitors with >=1 Purchase:    {txn_visitors:,} ({txn_visitors/total_visitors:.2%})")
print(f"View -> Cart Rate (Visitor):   {cart_visitors/view_visitors:.2%}")
print(f"Cart -> Purchase Rate (Visitor): {txn_visitors/cart_visitors:.2%}")
print(f"View -> Purchase Rate (Visitor): {txn_visitors/view_visitors:.2%}")

# 2. User Interaction Sparsity & Activity Distribution
user_activity = events.groupby("visitorid").size()
print("\n=== USER INTERACTION SPARSITY ===")
print(user_activity.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))
print(f"1 interaction only (extreme cold-start): {(user_activity == 1).mean():.2%}")
print(f"2-4 interactions (light browsers):       {((user_activity >= 2) & (user_activity <= 4)).mean():.2%}")
print(f">= 5 interactions (heavy browsers):      {(user_activity >= 5).mean():.2%}")

# 3. Check 80/20 Temporal Split Cutoff & Test Window Hit Coverage
cutoff_dt = events["dt"].quantile(0.80)
train_df = events[events["dt"] < cutoff_dt]
test_df = events[events["dt"] >= cutoff_dt]

# Relevant "hits" in test = addtocart or transaction
test_hits = test_df[test_df["event"].isin(["addtocart", "transaction"])]
test_hit_users = test_hits["visitorid"].nunique()
returning_hit_users = len(set(test_hits["visitorid"]).intersection(set(train_df["visitorid"])))

print("\n=== TEMPORAL TRAIN / TEST SPLIT (80 / 20 BY TIME) ===")
print(f"Cutoff Timestamp: {cutoff_dt}")
print(f"Train events: {len(train_df):,} ({len(train_df)/len(events):.1%}) | Users: {train_df['visitorid'].nunique():,}")
print(f"Test events:  {len(test_df):,} ({len(test_df)/len(events):.1%}) | Users: {test_df['visitorid'].nunique():,}")
print(f"Test days span: {(test_df['dt'].max() - cutoff_dt).days} days")
print(f"Test users with >=1 addtocart/transaction hit: {test_hit_users:,}")
print(f"  -> Returning users (seen in train): {returning_hit_users:,} ({returning_hit_users/test_hit_users:.2%})")
print(f"  -> New cold-start users (unseen in train): {test_hit_users - returning_hit_users:,} ({(test_hit_users - returning_hit_users)/test_hit_users:.2%})")

=== VISITOR-LEVEL CONVERSION FUNNEL ===
Total Unique Visitors:         1,407,580 (100.00%)
Visitors with >=1 View:        1,404,179 (99.76%)
Visitors with >=1 Add-to-Cart: 37,722 (2.68%)
Visitors with >=1 Purchase:    11,719 (0.83%)
View -> Cart Rate (Visitor):   2.69%
Cart -> Purchase Rate (Visitor): 31.07%
View -> Purchase Rate (Visitor): 0.83%

=== USER INTERACTION SPARSITY ===
count    1.407580e+06
mean     1.958042e+00
std      1.258049e+01
min      1.000000e+00
50%      1.000000e+00
75%      2.000000e+00
90%      3.000000e+00
95%      5.000000e+00
99%      1.300000e+01
max      7.757000e+03
dtype: float64
1 interaction only (extreme cold-start): 71.15%
2-4 interactions (light browsers):       23.05%
>= 5 interactions (heavy browsers):      5.80%

=== TEMPORAL TRAIN / TEST SPLIT (80 / 20 BY TIME) ===
Cutoff Timestamp: 2015-08-18 04:23:31.966000+00:00
Train events: 2,204,880 (80.0%) | Users: 1,123,767
Test events:  551,221 (20.0%) | Users: 308,549
Test days span: 30 days
Test users